In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="18IZGtu3zSMcAkfo94Ckw7iYEOSaQ6ygX", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_01_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# 🚀 Context Window Management: Taming the Finite Window

*Part 1 of the Vizuara series on Context Management & Reliability*
*Estimated time: 45 minutes*

Your language model's context window is not a filing cabinet where you can store everything. It is a desk with a hard size limit. Every token you put on that desk takes space away from something else. And worse — even within that desk, the model does not pay equal attention to everything.

In this notebook, we will build production-grade tools for structured extraction, context boundary placement, and "lost in the middle" mitigation. By the end, you will have a working context manager that extracts structured facts from documents, places critical information at attention-optimal positions, and visualizes exactly where the model's attention drops off.

In [ ]:
#@title 🎧 Listen: Why Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_why_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. 🤔 Why Does This Matter?

Consider this scenario: you are building an insurance claims agent. A customer submits a 50-page policy document, 3 prior conversations, and medical records. Your agent needs to process all of this, extract the right facts, and make a decision.

The naive approach is **progressive summarization** — summarize each chunk, then summarize the summaries. But each step is lossy. A policy exclusion on page 37 might be the single most important fact, and progressive summarization can silently drop it.

The better approach is **structured extraction** — pull out exactly the facts you need into a schema, preserving every critical detail with a link to its source.

Let us build both approaches and see the difference.

In [ ]:
# Setup: Install dependencies
!pip install matplotlib numpy -q

In [ ]:
import json
import random
import hashlib
import textwrap
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Clean plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'figure.dpi': 100,
})

random.seed(42)
np.random.seed(42)

print("✅ Setup complete. Let's manage some context windows!")

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. 💡 Building Intuition: Why Summarization Is Dangerous

Before we write any code, let us think about what happens when we summarize a document in stages.

Imagine a 50-page insurance policy. You read pages 1-10 and summarize them. Then pages 11-20, and summarize again. By the time you reach page 50, you have a compact summary — but each step had to decide what is "important enough" to keep.

The fundamental problem: **the summarizer does not know what question will be asked later.** A detail about pre-existing foundation cracks on page 7 seems minor when summarizing pages 1-10. But if the customer's claim is for foundation damage, that detail is everything.

### ✋ Stop and Think

Before scrolling down, ask yourself:
1. What kind of details are most likely to be dropped by summarization?
2. How would you design a system that *never* drops critical details?
3. What is the tradeoff between extraction and summarization?

*Take 2 minutes. Then scroll down.*

In [ ]:
#@title 🎧 Listen: Document Simulator
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_document_simulator.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. Building a Document Simulator

First, let us create a realistic document simulator. We need documents with buried critical details — just like real insurance policies.

In [ ]:
def generate_insurance_document(num_pages: int = 50, seed: int = 42) -> Dict:
    """
    Generate a simulated insurance policy document with buried critical details.

    The document has routine content on most pages, but critical exclusions
    and conditions scattered throughout — exactly the kind of thing
    progressive summarization would drop.
    """
    random.seed(seed)

    pages = []
    critical_details = []

    # Define critical details and where they appear
    hidden_facts = [
        {"page": 7, "paragraph": 3,
         "text": "Pre-existing foundation cracks are excluded from structural coverage",
         "category": "policy_exclusion"},
        {"page": 15, "paragraph": 2,
         "text": "Coverage limited to $25,000 for water damage from external sources only",
         "category": "coverage_limit"},
        {"page": 23, "paragraph": 5,
         "text": "Claims must be filed within 30 days of discovery, not occurrence",
         "category": "filing_deadline"},
        {"page": 37, "paragraph": 1,
         "text": "Mold remediation excluded if caused by ongoing maintenance failure",
         "category": "policy_exclusion"},
        {"page": 42, "paragraph": 4,
         "text": "Deductible increases to $5,000 for claims filed after initial 90-day period",
         "category": "deductible_change"},
    ]

    routine_content = [
        "Standard coverage applies to residential structures per policy terms.",
        "The insured party agrees to maintain the property in reasonable condition.",
        "Annual premium adjustments are calculated based on claims history.",
        "Property inspections may be conducted with 48-hour advance notice.",
        "Coverage extends to attached structures including garages and decks.",
        "Replacement cost valuation applies to dwelling coverage section.",
        "Additional living expenses covered up to 20% of dwelling coverage.",
        "Personal property coverage applies on an actual cash value basis.",
        "Liability coverage includes legal defense costs as supplementary.",
    ]

    for page_num in range(1, num_pages + 1):
        paragraphs = []
        num_paragraphs = random.randint(3, 6)

        for para_num in range(1, num_paragraphs + 1):
            # Check if this page/paragraph has a critical detail
            critical_fact = None
            for fact in hidden_facts:
                if fact["page"] == page_num and fact["paragraph"] == para_num:
                    critical_fact = fact
                    break

            if critical_fact:
                paragraphs.append(critical_fact["text"])
                critical_details.append({
                    "text": critical_fact["text"],
                    "category": critical_fact["category"],
                    "location": f"page {page_num}, paragraph {para_num}"
                })
            else:
                paragraphs.append(random.choice(routine_content))

        pages.append({
            "page_number": page_num,
            "paragraphs": paragraphs
        })

    return {
        "document_id": "POLICY-2024-78432",
        "title": "Homeowner's Insurance Policy - Standard Coverage",
        "total_pages": num_pages,
        "pages": pages,
        "ground_truth_critical_details": critical_details
    }


# Generate our test document
document = generate_insurance_document()
print(f"📄 Generated document: {document['title']}")
print(f"   Pages: {document['total_pages']}")
print(f"   Critical details hidden: {len(document['ground_truth_critical_details'])}")
print()
for detail in document['ground_truth_critical_details']:
    print(f"   ⚠️  [{detail['category']}] {detail['location']}")
    print(f"      \"{detail['text']}\"")

In [ ]:
#@title 🎧 Listen: Progressive Summarization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_progressive_summarization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. 🔧 Approach 1: Progressive Summarization (The Risky Way)

Let us implement progressive summarization and see how it loses information. We will simulate a summarizer that processes chunks and decides what to keep.

In [ ]:
def simulate_progressive_summarization(
    document: Dict,
    chunk_size: int = 10,
    retention_rate: float = 0.3
) -> Dict:
    """
    Simulate progressive summarization.

    Each chunk of pages gets summarized, retaining only a fraction of details.
    Critical details have a probability of being dropped at each stage.

    Args:
        document: The full document
        chunk_size: Number of pages per chunk
        retention_rate: Probability of retaining any given detail (0-1)

    Returns:
        Summary with tracking of what was kept and lost
    """
    pages = document["pages"]
    all_details_seen = []
    retained_details = []
    lost_details = []

    # Process in chunks
    for chunk_start in range(0, len(pages), chunk_size):
        chunk_end = min(chunk_start + chunk_size, len(pages))
        chunk_pages = pages[chunk_start:chunk_end]

        # Collect all details from this chunk
        chunk_details = []
        for page in chunk_pages:
            for para_idx, para in enumerate(page["paragraphs"]):
                # Check if this is a critical detail
                for truth in document["ground_truth_critical_details"]:
                    if para == truth["text"]:
                        chunk_details.append({
                            "text": para,
                            "source": f"page {page['page_number']}, paragraph {para_idx + 1}",
                            "category": truth["category"]
                        })

        all_details_seen.extend(chunk_details)

        # Simulate the summarizer deciding what to keep
        # Critical details look like routine text to the summarizer
        for detail in chunk_details:
            if random.random() < retention_rate:
                retained_details.append(detail)
            else:
                lost_details.append(detail)

    return {
        "method": "progressive_summarization",
        "total_critical_details": len(all_details_seen),
        "retained": retained_details,
        "lost": lost_details,
        "retention_rate": len(retained_details) / max(len(all_details_seen), 1)
    }


# Run progressive summarization multiple times to see the variance
print("Progressive Summarization Results (10 runs):\n")
print(f"{'Run':>4} {'Found':>6} {'Kept':>6} {'Lost':>6} {'Rate':>8}")
print("-" * 35)

retention_rates = []
for run in range(10):
    random.seed(run * 17 + 3)  # Different seed each run
    result = simulate_progressive_summarization(document, retention_rate=0.4)
    rate = result["retention_rate"]
    retention_rates.append(rate)
    print(f"{run+1:>4} {result['total_critical_details']:>6} "
          f"{len(result['retained']):>6} {len(result['lost']):>6} "
          f"{rate:>7.0%}")

print(f"\n📊 Average retention: {np.mean(retention_rates):.0%}")
print(f"   Worst case: {np.min(retention_rates):.0%}")
print(f"   Best case: {np.max(retention_rates):.0%}")

⚠️ **Key Insight:** Even with a 40% per-detail retention rate, progressive summarization frequently loses 2-3 out of 5 critical details. In a real insurance scenario, dropping a single policy exclusion could mean approving a claim that should have been denied — a potentially costly mistake.

In [ ]:
#@title 🎧 Listen: Structured Extraction
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_structured_extraction.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. 🔧 Approach 2: Structured Extraction (The Reliable Way)

Now let us build the structured extraction approach. Instead of asking "what is important?", we define exactly what fields we need and extract them systematically.

In [ ]:
# Define the extraction schema — what fields do we need?
EXTRACTION_SCHEMA = {
    "policy_exclusions": {
        "description": "Things the policy does NOT cover",
        "required_fields": ["text", "source_page", "source_paragraph"],
        "keywords": ["excluded", "not covered", "exception", "limitation", "excluded from"]
    },
    "coverage_limits": {
        "description": "Maximum amounts or caps on coverage",
        "required_fields": ["text", "limit_amount", "source_page", "source_paragraph"],
        "keywords": ["limited to", "maximum", "up to", "cap", "not exceed"]
    },
    "filing_deadlines": {
        "description": "Time limits for filing claims or taking action",
        "required_fields": ["text", "deadline_days", "source_page", "source_paragraph"],
        "keywords": ["within", "days", "deadline", "must be filed", "time limit"]
    },
    "deductible_changes": {
        "description": "Conditions that change the deductible amount",
        "required_fields": ["text", "amount", "condition", "source_page", "source_paragraph"],
        "keywords": ["deductible", "increases", "changes to", "adjusted"]
    }
}

print("📋 Extraction Schema Fields:")
for field, spec in EXTRACTION_SCHEMA.items():
    print(f"   {field}: {spec['description']}")
    print(f"      Keywords: {', '.join(spec['keywords'][:3])}...")

In [ ]:
def structured_extraction(document: Dict, schema: Dict) -> Dict:
    """
    Extract structured facts from a document using a defined schema.

    Instead of summarizing, we scan every page for specific categories
    of information. Nothing is lost because we never ask 'is this important?'
    — we ask 'does this match a category we care about?'

    Args:
        document: The full document
        schema: Extraction schema defining what to look for

    Returns:
        Structured extraction results with full provenance
    """
    results = {category: [] for category in schema}
    scan_log = []

    for page in document["pages"]:
        page_num = page["page_number"]

        for para_idx, paragraph in enumerate(page["paragraphs"]):
            para_lower = paragraph.lower()

            # Check each category's keywords
            for category, spec in schema.items():
                matched_keywords = [
                    kw for kw in spec["keywords"]
                    if kw.lower() in para_lower
                ]

                if matched_keywords:
                    extraction = {
                        "text": paragraph,
                        "source": {
                            "document_id": document["document_id"],
                            "page": page_num,
                            "paragraph": para_idx + 1,
                            "exact_quote": paragraph
                        },
                        "matched_keywords": matched_keywords,
                        "extraction_timestamp": datetime.now().isoformat(),
                        "confidence": min(0.95, 0.7 + 0.1 * len(matched_keywords))
                    }
                    results[category].append(extraction)
                    scan_log.append(f"Page {page_num}, Para {para_idx+1} → {category}")

    return {
        "method": "structured_extraction",
        "document_id": document["document_id"],
        "schema_used": list(schema.keys()),
        "results": results,
        "total_extractions": sum(len(v) for v in results.values()),
        "pages_scanned": document["total_pages"],
        "scan_log": scan_log
    }


# Run structured extraction
extraction = structured_extraction(document, EXTRACTION_SCHEMA)

print(f"🔍 Structured Extraction Results:\n")
print(f"   Pages scanned: {extraction['pages_scanned']}")
print(f"   Total extractions: {extraction['total_extractions']}")
print()

for category, items in extraction["results"].items():
    print(f"   📂 {category}: {len(items)} found")
    for item in items:
        src = item["source"]
        print(f"      • Page {src['page']}, Para {src['paragraph']}: "
              f"\"{item['text'][:60]}...\"")
        print(f"        Confidence: {item['confidence']:.2f} | "
              f"Keywords: {', '.join(item['matched_keywords'])}")

In [ ]:
#@title 🎧 Listen: Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. 📊 Visualization Checkpoint: Comparing Both Approaches

Let us visualize the critical difference between progressive summarization and structured extraction.

In [ ]:
def compare_approaches(document: Dict, num_runs: int = 50):
    """Run both approaches many times and compare detail retention."""

    # Progressive summarization — run many times
    prog_retained_counts = []
    for run in range(num_runs):
        random.seed(run * 13 + 7)
        result = simulate_progressive_summarization(
            document, retention_rate=0.4
        )
        prog_retained_counts.append(len(result["retained"]))

    # Structured extraction — deterministic, always finds everything
    ext_result = structured_extraction(document, EXTRACTION_SCHEMA)
    ext_count = ext_result["total_extractions"]
    total_critical = len(document["ground_truth_critical_details"])

    # Plot comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: histogram of progressive summarization results
    ax1 = axes[0]
    ax1.hist(prog_retained_counts, bins=range(0, total_critical + 2),
             color='#ff6b6b', edgecolor='white', alpha=0.8, align='left')
    ax1.axvline(x=total_critical, color='green', linestyle='--', linewidth=2,
                label=f'Target: {total_critical} details')
    ax1.set_xlabel('Critical Details Retained', fontsize=12)
    ax1.set_ylabel('Frequency (out of 50 runs)', fontsize=12)
    ax1.set_title('Progressive Summarization\n(Varies by Run)', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.set_xlim(-0.5, total_critical + 1)

    # Right: structured extraction (always gets everything)
    ax2 = axes[1]
    categories = list(ext_result["results"].keys())
    counts = [len(ext_result["results"][cat]) for cat in categories]
    short_labels = [cat.replace('_', '\n') for cat in categories]
    bars = ax2.bar(short_labels, counts, color='#51cf66', edgecolor='white', alpha=0.8)
    ax2.set_ylabel('Items Extracted', fontsize=12)
    ax2.set_title('Structured Extraction\n(Deterministic — Same Every Time)', fontsize=13, fontweight='bold')

    # Add count labels on bars
    for bar, count in zip(bars, counts):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(count), ha='center', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.savefig('approach_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n📊 Progressive Summarization: retained {np.mean(prog_retained_counts):.1f} "
          f"out of {total_critical} details on average (worst: {min(prog_retained_counts)})")
    print(f"📊 Structured Extraction: retained ALL relevant items deterministically")


compare_approaches(document)

In [ ]:
#@title 🎧 Listen: Sandwich Pattern
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_sandwich_pattern.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. 🔧 The Sandwich Pattern: Placing Critical Info at Context Boundaries

Now let us tackle the "lost in the middle" effect. Research by Liu et al. (2023) showed that language models pay stronger attention to information at the **beginning** and **end** of the context, with a U-shaped attention curve.

We will build a context builder that places critical information at optimal positions.

In [ ]:
def simulate_attention_curve(context_length: int) -> np.ndarray:
    """
    Simulate the U-shaped attention curve from 'Lost in the Middle' research.

    Models attend most strongly to the beginning and end of context,
    with a dip in the middle.
    """
    positions = np.linspace(0, 1, context_length)

    # U-shaped curve: high at edges, low in middle
    # Based on the empirical findings from Liu et al. (2023)
    attention = 0.4 + 0.6 * (4 * (positions - 0.5)**2)

    # Add some noise for realism
    noise = np.random.normal(0, 0.03, context_length)
    attention = np.clip(attention + noise, 0.1, 1.0)

    return attention


# Visualize the attention curve
fig, ax = plt.subplots(figsize=(12, 4))

context_length = 100
attention = simulate_attention_curve(context_length)
positions = np.arange(context_length)

ax.fill_between(positions, attention, alpha=0.3, color='steelblue')
ax.plot(positions, attention, color='steelblue', linewidth=2)

# Mark the zones
ax.axvspan(0, 15, alpha=0.15, color='green', label='High attention (start)')
ax.axvspan(35, 65, alpha=0.15, color='red', label='Low attention (middle)')
ax.axvspan(85, 100, alpha=0.15, color='green', label='High attention (end)')

ax.set_xlabel('Position in Context Window', fontsize=12)
ax.set_ylabel('Relative Attention Strength', fontsize=12)
ax.set_title('The "Lost in the Middle" Effect — Attention Is U-Shaped',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, context_length - 1)

plt.tight_layout()
plt.savefig('attention_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 Key insight: Information placed in the middle of the context window")
print("   receives significantly less attention than information at the edges.")

In [ ]:
class SandwichContextBuilder:
    """
    Build context with the 'Sandwich Pattern' — critical information
    placed at the beginning AND end of the context for maximum attention.

    Structure:
    1. System instructions + critical rules (HIGH ATTENTION)
    2. Extracted critical facts (HIGH ATTENTION)
    3. Supporting documentation (medium attention — middle zone)
    4. Reminder of critical rules (HIGH ATTENTION)
    5. User query (HIGH ATTENTION)
    """

    def __init__(self, system_instructions: str):
        self.system_instructions = system_instructions
        self.critical_facts = []
        self.supporting_docs = []
        self.token_budget = 0

    def add_critical_fact(self, fact: Dict):
        """Add a fact that MUST be attended to."""
        self.critical_facts.append(fact)

    def add_supporting_doc(self, doc: str, source: str):
        """Add background documentation (goes in the middle)."""
        self.supporting_docs.append({"text": doc, "source": source})

    def build(self, user_query: str) -> str:
        """
        Assemble the context with critical info at the boundaries.
        """
        sections = []

        # === TOP (high attention zone) ===
        sections.append("=== SYSTEM INSTRUCTIONS ===")
        sections.append(self.system_instructions)
        sections.append("")

        # Critical facts at the top
        sections.append("=== CRITICAL FACTS — READ CAREFULLY ===")
        for fact in self.critical_facts:
            sections.append(f"• [{fact.get('category', 'IMPORTANT')}] {fact['text']}")
            if 'source' in fact:
                src = fact['source']
                sections.append(f"  (Source: {src.get('document_id', 'N/A')}, "
                              f"page {src.get('page', 'N/A')})")
        sections.append("")

        # === MIDDLE (lower attention zone) ===
        sections.append("=== SUPPORTING DOCUMENTATION ===")
        for doc in self.supporting_docs:
            sections.append(f"[{doc['source']}]")
            sections.append(doc["text"])
            sections.append("")
        sections.append("=== END SUPPORTING DOCUMENTATION ===")
        sections.append("")

        # === BOTTOM (high attention zone) ===
        # Repeat critical facts at the bottom!
        sections.append("=== REMINDER — CRITICAL FACTS (re-read before answering) ===")
        for fact in self.critical_facts:
            sections.append(f"• [{fact.get('category', 'IMPORTANT')}] {fact['text']}")
        sections.append("")

        sections.append(f"=== USER QUERY ===")
        sections.append(user_query)

        context = "\n".join(sections)
        self.token_budget = len(context.split())  # Rough word-based estimate
        return context

    def get_placement_report(self) -> Dict:
        """Show where each piece of information is placed."""
        return {
            "top_zone": {
                "contents": ["system_instructions", "critical_facts"],
                "attention": "HIGH"
            },
            "middle_zone": {
                "contents": [f"supporting_doc: {d['source']}" for d in self.supporting_docs],
                "attention": "MEDIUM-LOW"
            },
            "bottom_zone": {
                "contents": ["critical_facts_reminder", "user_query"],
                "attention": "HIGH"
            }
        }


# Build a context using our structured extraction results
builder = SandwichContextBuilder(
    system_instructions=(
        "You are an insurance claims analyst. You MUST check all policy exclusions "
        "before approving any claim. NEVER approve a claim that matches an exclusion."
    )
)

# Add extracted critical facts
for category, items in extraction["results"].items():
    for item in items:
        builder.add_critical_fact({
            "text": item["text"],
            "category": category,
            "source": item["source"]
        })

# Add some supporting documentation (goes in the middle)
builder.add_supporting_doc(
    "Customer filed claim on March 5, 2024 for water damage to basement.",
    "claim_form.pdf"
)
builder.add_supporting_doc(
    "Contractor estimates $45,000 in repairs. Adjuster visited March 12.",
    "adjuster_report.pdf"
)

# Build the context
context = builder.build("Should we approve claim CLM-2024-78432 for basement water damage?")

print("📋 Context Placement Report:")
report = builder.get_placement_report()
for zone, info in report.items():
    print(f"\n   {zone.upper()} (Attention: {info['attention']}):")
    for item in info['contents']:
        print(f"      • {item}")

print(f"\n📊 Total context size: ~{builder.token_budget} words")
print(f"\n--- First 500 chars of built context ---")
print(context[:500])
print("...")
print(f"\n--- Last 500 chars of built context ---")
print(context[-500:])

In [ ]:
#@title 🎧 Listen: Todo Exercise
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_todo_exercise.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. 🔧 Your Turn: Build a Context Placement Optimizer

Now it is your turn. Implement the `optimize_placement` method that scores different placement strategies and picks the best one.

In [ ]:
def score_placement(context_sections: List[Dict], attention_curve: np.ndarray) -> float:
    """
    Score a placement strategy by how well critical information
    aligns with high-attention positions.

    Args:
        context_sections: List of {"text": ..., "is_critical": bool, "position": int}
        attention_curve: The U-shaped attention weights

    Returns:
        Score (higher is better) — sum of attention weights for critical items
    """
    # ============ TODO ============
    # Step 1: Filter to only critical sections
    # Step 2: For each critical section, look up its attention weight
    # Step 3: Sum the attention weights and return the total
    # ==============================

    score = 0.0  # YOUR CODE HERE

    return score

In [ ]:
# ✅ Verification: Run this cell to check your implementation

# Create test data
test_sections = [
    {"text": "Critical rule 1", "is_critical": True, "position": 0},
    {"text": "Background info", "is_critical": False, "position": 5},
    {"text": "More background", "is_critical": False, "position": 10},
    {"text": "Critical rule 2", "is_critical": True, "position": 19},
]

test_attention = np.array([1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.3, 0.3,
                           0.3, 0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.6, 0.7, 0.9])

result = score_placement(test_sections, test_attention)

# Critical items at positions 0 (attention=1.0) and 19 (attention=0.9) → score = 1.9
expected = 1.9
assert abs(result - expected) < 0.01, \
    f"❌ Expected score {expected}, got {result}. Make sure you're summing attention at critical positions."
print(f"✅ Correct! Score = {result:.2f}")
print("   Critical info at positions 0 and 19 → high attention weights → high score!")

In [ ]:
#@title 🎧 Listen: Final System
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_final_system.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. 🎯 Final System: Complete Context Manager

Let us put everything together into a complete context management system.

In [ ]:
class ContextManager:
    """
    Production-grade context manager that:
    1. Extracts structured facts from documents
    2. Places critical information at context boundaries (Sandwich Pattern)
    3. Tracks what was extracted and where it came from
    4. Provides a complete audit trail
    """

    def __init__(self, schema: Dict, system_prompt: str, max_tokens: int = 128000):
        self.schema = schema
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.extraction_history = []

    def process_document(self, document: Dict) -> Dict:
        """Extract structured facts from a document."""
        result = structured_extraction(document, self.schema)
        self.extraction_history.append({
            "document_id": document["document_id"],
            "timestamp": datetime.now().isoformat(),
            "extractions": result["total_extractions"]
        })
        return result

    def build_optimized_context(self, extraction: Dict, query: str) -> Dict:
        """Build context with optimal information placement."""
        builder = SandwichContextBuilder(self.system_prompt)

        # Add all extracted facts as critical
        all_facts = []
        for category, items in extraction["results"].items():
            for item in items:
                fact = {
                    "text": item["text"],
                    "category": category,
                    "source": item["source"],
                    "confidence": item["confidence"]
                }
                builder.add_critical_fact(fact)
                all_facts.append(fact)

        context = builder.build(query)

        return {
            "context": context,
            "word_count": len(context.split()),
            "facts_included": len(all_facts),
            "placement": builder.get_placement_report(),
            "audit_trail": {
                "extraction_method": "structured_extraction",
                "schema_fields": list(self.schema.keys()),
                "facts": all_facts,
                "query": query,
                "timestamp": datetime.now().isoformat()
            }
        }

    def generate_report(self) -> str:
        """Generate a human-readable report of all processing."""
        lines = ["=" * 60]
        lines.append("CONTEXT MANAGER — PROCESSING REPORT")
        lines.append("=" * 60)
        lines.append(f"Documents processed: {len(self.extraction_history)}")
        lines.append(f"Total extractions: "
                    f"{sum(h['extractions'] for h in self.extraction_history)}")
        lines.append(f"Schema fields: {', '.join(self.schema.keys())}")
        lines.append("")

        for entry in self.extraction_history:
            lines.append(f"  📄 {entry['document_id']} — "
                        f"{entry['extractions']} extractions "
                        f"({entry['timestamp'][:19]})")

        lines.append("=" * 60)
        return "\n".join(lines)


# Run the full system
manager = ContextManager(
    schema=EXTRACTION_SCHEMA,
    system_prompt=(
        "You are an insurance claims analyst. Check all policy exclusions "
        "before approving. Flag any claims that match exclusions."
    )
)

# Process the document
extraction = manager.process_document(document)

# Build optimized context for a specific query
result = manager.build_optimized_context(
    extraction,
    "Should we approve claim CLM-2024-78432 for basement water damage "
    "caused by foundation cracks?"
)

print(manager.generate_report())
print(f"\n🎯 Built context: {result['word_count']} words, "
      f"{result['facts_included']} critical facts")
print(f"\n📋 Audit trail available with {len(result['audit_trail']['facts'])} "
      f"traced facts")

# Show a critical fact with full provenance
if result['audit_trail']['facts']:
    fact = result['audit_trail']['facts'][0]
    print(f"\n🔍 Example traced fact:")
    print(f"   Text: \"{fact['text']}\"")
    print(f"   Category: {fact['category']}")
    print(f"   Source: {fact['source']['document_id']}, "
          f"page {fact['source']['page']}")
    print(f"   Confidence: {fact['confidence']:.2f}")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 10. 🤔 Reflection Questions and Extensions

1. **What happens if the extraction schema misses a keyword?** How would you make the schema adaptive — learning from documents it has already processed?

2. **The Sandwich Pattern duplicates critical information.** This uses more tokens. When would the tradeoff not be worth it?

3. **Try modifying the attention curve.** What if the model had a flat attention curve (equal everywhere)? Would the Sandwich Pattern still help?

4. **Extension challenge:** Implement a `DynamicSchemaUpdater` that monitors extraction results and suggests new keywords to add to the schema based on patterns it sees in documents.

5. **Real-world consideration:** How would you handle documents in multiple languages? What changes would the extraction schema need?